# BGL campaign — Google Colab

Ten notebook runs the corrected, frozen BGL research campaign: **25 representation/LLM runs + 15 PB3 runs + 5 Isolation Forest runs = 45 results**.

Launch flags:

- **`RUN_PREPARE`** — build five representation graphs and three PB3 graphs using the same BGL source, frozen LLM enrichment and `split_lock.npz`.
- **`RUN_TRAIN`** — train the 8 GAE graph arms across five seeds (40 GAE runs).
- **`RUN_BASELINE`** — run the 5-seed Isolation Forest baseline on the prepared `hybrid_llm` graph bundle.

The campaign uses a chronological BGL split, disjoint 20-minute windows, 20-minute embargo at both split boundaries, train-only Drain/TF-IDF fitting, and validation-only checkpoint/threshold selection. The primary LLM contrast is `hybrid_llm` versus `hybrid_raw`; the primary PB3 contrast is `hybrid_llm` versus `no_temporal_or_positional_features`.

**Before running**

1. Push the current implementation to the branch selected below; this notebook requires the updated `ablation_bgl_llm_closed.yaml`, `ablation_bgl_pb3.yaml`, Isolation Forest mode, and provenance checks.
2. Upload `BGL_full.log` to `MyDrive/hybrid-log-analyzer-artifacts/data/raw/bgl/BGL_full.log`.
3. Add Colab secrets (🔑): `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_ENDPOINT`, and `AZURE_OPENAI_DEPLOYMENT_DEEPSEEK_V4_PRO`.
4. Use `SMOKE=True` only for a technical pilot; do not publish its results. For final results use `SMOKE=False`.

The same `CAMPAIGN_ID` must be used for preparation, training, baseline, and reporting. Do not reuse an old campaign directory created before the current protocol.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print(f"Running in Google Colab: {IN_COLAB}")

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPOSITORY_URL = "https://github.com/michaail/hybrid-logs-analyzer-research.git"
GIT_REF = "ablation-cursor"  # must contain the current campaign implementation
DATASET = "bgl"
CAMPAIGN_ID = "bgl_llm_pb3_20260921"  # use a new ID for the corrected protocol
RUN_PREPARE = True
RUN_TRAIN = False
RUN_BASELINE = False
FAMILY = "A"
SMOKE = True  # technical pilot only; set False for the final 45-run campaign
REUSE_DRIVE_CACHE = True
AZURE_SECRET_KEYS = (
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_DEPLOYMENT_DEEPSEEK_V4_PRO",
)
REPRESENTATION_MATRIX = "configs/ablation_bgl_llm_closed.yaml"
PB3_MATRIX = "configs/ablation_bgl_pb3.yaml"
CAMPAIGN_RELATIVE_DIR = f"campaigns/{CAMPAIGN_ID}"
BGL_RAW_RELATIVE = "data/raw/bgl/BGL_full.log"


def find_local_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "modules" / "models" / "gae.py").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from the hybrid-logs-analyzer-research checkout.")


REPO_ROOT = Path("/content/hybrid-logs-analyzer-research") if IN_COLAB else find_local_repository_root()
WORKSPACE_ROOT = Path("/content/workspace") if IN_COLAB else REPO_ROOT
DRIVE_ARTIFACT_ROOT = Path("/content/drive/MyDrive/hybrid-log-analyzer-artifacts")


def final_drive_sync() -> None:
    if not IN_COLAB:
        return
    for name in ("artifacts", "models", "outputs", "runs", "campaigns"):
        source = WORKSPACE_ROOT / name
        if source.exists():
            destination = DRIVE_ARTIFACT_ROOT / name
            destination.mkdir(parents=True, exist_ok=True)
            subprocess.run(["rsync", "-a", "--partial", f"{source}/", f"{destination}/"], check=True)


def load_azure_credentials() -> None:
    missing = []
    userdata_get = None
    if IN_COLAB:
        from google.colab import userdata
        userdata_get = userdata.get
    for key in AZURE_SECRET_KEYS:
        if os.environ.get(key):
            continue
        value = None
        if userdata_get is not None:
            try:
                value = userdata_get(key)
            except Exception:
                value = None
        if value:
            os.environ[key] = str(value)
        else:
            missing.append(key)
    if missing:
        raise EnvironmentError(
            "Azure credentials missing: " + ", ".join(missing)
            + ". Add them as Colab secrets (🔑) with notebook access."
        )


if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not REPO_ROOT.exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1", "--branch", GIT_REF,
            REPOSITORY_URL, str(REPO_ROOT),
        ])
    else:
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", GIT_REF])
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "checkout", "-B", GIT_REF, "FETCH_HEAD"])
    DRIVE_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
    WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ["PIPELINE_WORKSPACE_ROOT"] = str(WORKSPACE_ROOT)

    def stage_from_drive(relative_path: str) -> Path:
        source = DRIVE_ARTIFACT_ROOT / relative_path
        destination = WORKSPACE_ROOT / relative_path
        if not source.exists():
            raise FileNotFoundError(f"Missing Drive artifact: {source}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        if source.is_dir():
            subprocess.check_call(["rsync", "-a", "--partial", f"{source}/", f"{destination}/"])
        else:
            shutil.copy2(source, destination)
        return destination

    def stage_tree_from_drive(relative_path: str) -> None:
        source = DRIVE_ARTIFACT_ROOT / relative_path
        if source.exists():
            destination = WORKSPACE_ROOT / relative_path
            destination.mkdir(parents=True, exist_ok=True)
            subprocess.check_call(["rsync", "-a", "--partial", f"{source}/", f"{destination}/"])

    def stage_bgl_raw_from_drive() -> Path:
        source = DRIVE_ARTIFACT_ROOT / BGL_RAW_RELATIVE
        if not source.exists():
            raise FileNotFoundError(f"Upload BGL_full.log to {source}.")
        return stage_from_drive(BGL_RAW_RELATIVE)

    if REUSE_DRIVE_CACHE:
        for relative_path in (f"artifacts/cache/{DATASET}", "artifacts/runs", f"outputs/{DATASET}", "campaigns"):
            stage_tree_from_drive(relative_path)
else:
    def stage_from_drive(relative_path: str) -> Path:
        return WORKSPACE_ROOT / relative_path
    def stage_tree_from_drive(relative_path: str) -> None:
        del relative_path
    def stage_bgl_raw_from_drive() -> Path:
        return WORKSPACE_ROOT / BGL_RAW_RELATIVE


CAMPAIGN_DIR = (WORKSPACE_ROOT / CAMPAIGN_RELATIVE_DIR) if (WORKSPACE_ROOT / CAMPAIGN_RELATIVE_DIR / "manifest.json").exists() else None
RAW_LOG_PATH = WORKSPACE_ROOT / BGL_RAW_RELATIVE
if RUN_PREPARE:
    load_azure_credentials()
    RAW_LOG_PATH = stage_bgl_raw_from_drive()
    CAMPAIGN_DIR = WORKSPACE_ROOT / CAMPAIGN_RELATIVE_DIR
if (RUN_TRAIN or RUN_BASELINE) and CAMPAIGN_DIR is None:
    raise FileNotFoundError("Training/baseline requires a prepared campaign manifest on Drive or locally.")
if not any((RUN_PREPARE, RUN_TRAIN, RUN_BASELINE)):
    raise ValueError("Enable RUN_PREPARE, RUN_TRAIN, or RUN_BASELINE.")

print(f"Code checkout : {REPO_ROOT}")
print(f"Workspace     : {WORKSPACE_ROOT}")
print(f"Campaign dir  : {CAMPAIGN_DIR}")
print(f"Prepare/train/baseline: {RUN_PREPARE}/{RUN_TRAIN}/{RUN_BASELINE}")
print(f"Raw log       : {RAW_LOG_PATH}")

In [ ]:
# Safety default: this notebook produces publishable full-study results.
# Set to True only for a one-epoch pipeline diagnostic.
SMOKE = False
print(f"Smoke training : {SMOKE}")

In [ ]:
if IN_COLAB:
    subprocess.check_call([
        sys.executable, str(REPO_ROOT / "scripts" / "install_colab.py"),
        "--project-root", str(REPO_ROOT),
    ])
else:
    print("Local environment — use this repo venv / requirements.txt, not requirements-colab.txt.")

In [ ]:
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
os.environ["PIPELINE_WORKSPACE_ROOT"] = str(WORKSPACE_ROOT)

gae_module = REPO_ROOT / "src" / "modules" / "models" / "gae.py"
if not gae_module.exists():
    raise FileNotFoundError(
        f"Missing {gae_module}. This Colab clone of {GIT_REF!r} does not contain "
        "the GAE package. Commit and push src/modules/models/, then "
        "Runtime → Restart session and rerun from the clone cell."
    )

import torch
print(f"Working directory: {Path.cwd()}")
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()} | devices: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

## Prepare the corrected BGL campaign

This builds the five representation/LLM graph arms first and the three PB3 arms second, into one campaign directory. The second preparation reuses the frozen `split_lock.npz`, parser, sequences and LLM-enrichment cache.

Use a new `CAMPAIGN_ID`. Preparation does not train models. For the final campaign, run this once with `SMOKE=False`; `SMOKE` affects only the later training cell.

In [ ]:
if not RUN_PREPARE:
    print("Skipping preparation (RUN_PREPARE=False).")
else:
    matrices = (REPRESENTATION_MATRIX, PB3_MATRIX)
    for matrix_relative in matrices:
        matrix_path = REPO_ROOT / matrix_relative
        prepare_command = [
            sys.executable,
            str(REPO_ROOT / "scripts" / "prepare_bgl_campaign.py"),
            "--campaign-id", CAMPAIGN_ID,
            "--workspace-root", str(WORKSPACE_ROOT),
            "--code-root", str(REPO_ROOT),
            "--config", str(REPO_ROOT / "configs" / "ablation_base.yaml"),
            "--matrix", str(matrix_path),
            "--campaign-dir", str(WORKSPACE_ROOT / CAMPAIGN_RELATIVE_DIR),
        ]
        if IN_COLAB:
            prepare_command.extend(["--checkpoint-root", str(DRIVE_ARTIFACT_ROOT)])
        print(" ".join(str(part) for part in prepare_command))
        try:
            completed = subprocess.run(prepare_command, check=False)
        finally:
            final_drive_sync()
        if completed.returncode:
            raise RuntimeError(f"BGL preparation failed for {matrix_relative}: {completed.returncode}")
    CAMPAIGN_DIR = WORKSPACE_ROOT / CAMPAIGN_RELATIVE_DIR
    print(f"Preparation complete: {CAMPAIGN_DIR}")
    manifest_path = CAMPAIGN_DIR / "manifest.json"
    manifest = json.loads(manifest_path.read_text())
    print("Prepared graph arms:", [item["name"] for item in manifest.get("graphs", [])])

## Train 40 GAE runs

Set `RUN_TRAIN=True`, `RUN_PREPARE=False`, `RUN_BASELINE=False`, `FAMILY="A"`, and `SMOKE=False` after the pilot passes. The runner trains every prepared graph arm in the manifest across five seeds: 25 representation/LLM runs and 15 PB3 runs.

It is resumable: completed seed/arm runs with compatible configuration are skipped.

In [ ]:
BASE_CONFIG_PATH = REPO_ROOT / "configs" / "ablation_base.yaml"
if not RUN_TRAIN:
    print("Skipping GAE training (RUN_TRAIN=False).")
else:
    if FAMILY.upper() != "A":
        raise ValueError("The corrected BGL campaign trains the prepared Family A arms; set FAMILY='A'.")
    runner_command = [
        sys.executable, str(REPO_ROOT / "run_ablation.py"),
        "--mode", "train-only",
        "--config", str(BASE_CONFIG_PATH),
        "--workspace-root", str(WORKSPACE_ROOT),
        "--code-root", str(REPO_ROOT),
        "--campaign-id", CAMPAIGN_ID,
        "--campaign-dir", str(CAMPAIGN_DIR),
        "--family", "A",
        "--set", "experiment.dataset=bgl",
    ]
    if SMOKE:
        runner_command.extend([
            "--set", "training.test_run=true",
            "--set", "training.epochs=1",
            "--set", "training.test_samples=5000",
        ])
    if IN_COLAB:
        runner_command.extend(["--checkpoint-root", str(DRIVE_ARTIFACT_ROOT)])
    print(" ".join(str(part) for part in runner_command))
    try:
        completed = subprocess.run(runner_command, check=False)
    finally:
        final_drive_sync()
    if completed.returncode:
        raise RuntimeError(f"GAE campaign training failed with exit code {completed.returncode}")
    print("GAE campaign training finished.")

## Run the 5 Isolation Forest baseline results

After a prepared campaign exists, set `RUN_BASELINE=True`. This uses the frozen `hybrid_llm` graph bundle solely to obtain the identical BGL windows and split; it does not use LLM text or GNN embeddings as baseline features. Set `SMOKE=False` for final results.

In [ ]:
if not RUN_BASELINE:
    print("Skipping Isolation Forest baseline (RUN_BASELINE=False).")
else:
    if SMOKE:
        raise ValueError("Isolation Forest pilot is not implemented separately; use SMOKE=False for the final baseline.")
    baseline_command = [
        sys.executable, str(REPO_ROOT / "run_ablation.py"),
        "--mode", "isolation-forest",
        "--config", str(BASE_CONFIG_PATH),
        "--workspace-root", str(WORKSPACE_ROOT),
        "--code-root", str(REPO_ROOT),
        "--campaign-id", CAMPAIGN_ID,
        "--campaign-dir", str(CAMPAIGN_DIR),
        "--set", "experiment.dataset=bgl",
    ]
    if IN_COLAB:
        baseline_command.extend(["--checkpoint-root", str(DRIVE_ARTIFACT_ROOT)])
    print(" ".join(str(part) for part in baseline_command))
    try:
        completed = subprocess.run(baseline_command, check=False)
    finally:
        final_drive_sync()
    if completed.returncode:
        raise RuntimeError(f"Isolation Forest failed with exit code {completed.returncode}")
    print("Isolation Forest baseline finished.")

## Leaderboard and comparison plots

Loaded from disk (not notebook RAM) so a later session can replay without retraining.

In [ ]:
import pandas as pd
from IPython.display import Image, display

report_root = DRIVE_ARTIFACT_ROOT if IN_COLAB else WORKSPACE_ROOT
campaign_report = report_root / "outputs" / DATASET / "campaigns" / CAMPAIGN_ID
if not RUN_TRAIN and not (campaign_report / "leaderboard.csv").exists():
    print("No leaderboard yet. After prepare, set RUN_TRAIN=True and rerun train.")
else:
    if not (campaign_report / "leaderboard.json").exists():
        subprocess.check_call([
            sys.executable, str(REPO_ROOT / "run_ablation.py"),
            "--mode", "report",
            "--config", str(BASE_CONFIG_PATH),
            "--workspace-root", str(WORKSPACE_ROOT),
            "--code-root", str(REPO_ROOT),
            "--campaign-id", CAMPAIGN_ID,
            "--set", f"experiment.dataset={DATASET}",
        ])
        if IN_COLAB:
            final_drive_sync()
        campaign_report = WORKSPACE_ROOT / "outputs" / DATASET / "campaigns" / CAMPAIGN_ID

    leaderboard = pd.read_csv(campaign_report / "leaderboard.csv")
    display(leaderboard)
    print((campaign_report / "README.md").read_text())
    for figure_name in ("ablation_comparison.png", "loss_curves.png", "component_comparison.png"):
        path = campaign_report / figure_name
        if path.exists():
            display(Image(filename=str(path)))

## Replay a previous campaign

Set `CAMPAIGN_ID` above and run only the leaderboard cell. Per-run packs live in `outputs/bgl/<campaign_id>_<arm>/figures/`.

In [ ]:
from IPython.display import Image, display

runs_root = (DRIVE_ARTIFACT_ROOT if IN_COLAB else WORKSPACE_ROOT) / "outputs" / DATASET
for run_dir in sorted(runs_root.glob(f"{CAMPAIGN_ID}_*")):
    if not (run_dir / "metrics.json").exists():
        continue
    metrics = json.loads((run_dir / "metrics.json").read_text())
    print(f"{run_dir.name}: F1={metrics.get('test_f1')} PR-AUC={metrics.get('test_pr_auc')} ROC-AUC={metrics.get('test_roc_auc')}")
    preview = run_dir / "figures" / "test_pr_roc.png"
    if preview.exists():
        display(Image(filename=str(preview)))